# Route Optimization (VRP) — OR-Tools

Vehicle Routing Problem: multiple vehicles, multiple delivery locations, capacity constraints, time windows.

## Problem Definition
- n locations (depot + delivery points)
- m vehicles with capacity constraints
- Distance matrix (euclidean or real road distances)
- Optional: time windows, vehicle-specific costs

## Visualization
Folium map showing optimal routes.

In [ ]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import numpy as np
import folium

# Sample distance matrix (4 locations: depot + 3 delivery points)
distance_matrix = [
    [0, 10, 20, 30],
    [10, 0, 15, 25],
    [20, 15, 0, 10],
    [30, 25, 10, 0]
]

# Data
data = {
    'distance_matrix': distance_matrix,
    'num_vehicles': 2,
    'depot': 0
}

# Create routing model
manager = pywrapcp.RoutingIndexManager(
    len(data['distance_matrix']),
    data['num_vehicles'],
    data['depot']
)

routing = pywrapcp.RoutingModel(manager)

# Transit callback
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Search parameters
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

# Solve
solution = routing.SolveWithParameters(search_parameters)

if solution:
    print("Solution found!")
    total_distance = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route = []
        route_distance = 0
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
            prev_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(prev_index, index, vehicle_id)
        route.append(manager.IndexToNode(index))
        print(f"\nVehicle {vehicle_id} route: {route}")
        print(f"Distance: {route_distance}")
        total_distance += route_distance
    print(f"\nTotal distance: {total_distance}")
else:
    print("No solution found!")